# Iris flower classification with a Decision Tree

**Case study:** predict Iris-setosa, Iris-versicolor or Iris-virginica from four flower measurements in centimetres.

Dataset: [Kaggle UCI Iris](https://www.kaggle.com/datasets/uciml/iris), downloaded directly from Kaggle. 150 rows, three equally represented classes. Exclude the row identifier. Reserve a stratified 20% test set before fitting. This notebook runs in a Jupyter kernel, independently of VS Code.

The bundled CSV makes reruns reproducible without Kaggle credentials. The download URL and SHA-256 are recorded alongside the model. Small educational data; holdout accuracy is not a guarantee on new real-world flowers.

In [1]:
from pathlib import Path
import hashlib, json, platform
from datetime import datetime, timezone
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ROOT = Path.cwd() if (Path.cwd() / 'data' / 'Iris.csv').exists() else Path.cwd().parent
DATA = ROOT / 'data' / 'Iris.csv'
FEATURES = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
df = pd.read_csv(DATA)
assert df.shape == (150, 6)
assert not df.isna().any().any()
assert set(df['Species']) == {'Iris-setosa', 'Iris-versicolor', 'Iris-virginica'}
print('Shape:', df.shape)
print('Class counts:', df['Species'].value_counts().to_dict())
df.head()

Shape: (150, 6)
Class counts: {'Iris-setosa': 50, 'Iris-versicolor': 50, 'Iris-virginica': 50}


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## Explore the data
Summarize the measurement ranges and class distribution. No hyperparameters are selected using the held-out test set.

In [2]:
df[FEATURES].describe()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


## Train and test
A fixed random seed and stratified split preserve 40 training / 10 test examples per class. A depth-limited decision tree uses entropy to choose splits on the original measurement scale. The row ID and target are excluded from the features.

In [3]:
X = df[FEATURES].to_numpy()
y = df['Species'].to_numpy()
X_train, X_test, y_train, y_test, train_ids, test_ids = train_test_split(
    X, y, df['Id'].to_numpy(), test_size=0.2, random_state=42, stratify=y)
assert set(train_ids).isdisjoint(test_ids)
model = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = float(accuracy_score(y_test, predictions))
report = classification_report(y_test, predictions, output_dict=True, zero_division=0)
metrics = {'accuracy': accuracy, 'macro_f1': report['macro avg']['f1-score'],
           'training_rows': len(X_train), 'test_rows': len(X_test),
           'classification_report': report,
           'confusion_matrix': confusion_matrix(y_test, predictions, labels=model.classes_).tolist()}
print(f'Held-out accuracy: {accuracy:.4f}')
print(classification_report(y_test, predictions, zero_division=0))
print('Confusion matrix (class order):', model.classes_.tolist())
print(np.asarray(metrics['confusion_matrix']))
assert accuracy >= 0.8

Held-out accuracy: 0.9667
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30

Confusion matrix (class order): ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


## Save model, parameters and evidence
The joblib bundle contains the fitted estimator, exact hyperparameters, feature order, class names, metrics and data checksum. JSON files make the evidence readable without loading pickle. Load joblib files only from trusted sources.

In [4]:
sha256 = hashlib.sha256(DATA.read_bytes()).hexdigest()
artifact = {'model': model, 'parameters': model.get_params(), 'features': FEATURES,
            'classes': model.classes_.tolist(), 'metrics': metrics,
            'dataset_sha256': sha256, 'sklearn_version': sklearn.__version__}
(ROOT / 'models').mkdir(exist_ok=True)
(ROOT / 'reports').mkdir(exist_ok=True)
joblib.dump(artifact, ROOT / 'models' / 'model.joblib')
(ROOT / 'models' / 'parameters.json').write_text(json.dumps(model.get_params(), indent=2))
(ROOT / 'reports' / 'metrics.json').write_text(json.dumps(metrics, indent=2))
provenance = {'source': 'https://www.kaggle.com/datasets/uciml/iris',
 'download_url': 'https://www.kaggle.com/api/v1/datasets/download/uciml/iris',
 'csv_sha256': sha256, 'split_random_state': 42, 'stratified': True,
 'train_ids': train_ids.tolist(), 'test_ids': test_ids.tolist(),
 'python_version': platform.python_version(), 'sklearn_version': sklearn.__version__,
 'executed_at_utc': datetime.now(timezone.utc).isoformat()}
(ROOT / 'reports' / 'training_provenance.json').write_text(json.dumps(provenance, indent=2))
restored = joblib.load(ROOT / 'models' / 'model.joblib')
assert np.array_equal(restored['model'].predict(X_test), predictions)
assert restored['parameters'] == model.get_params()
print('Saved and reloaded models/model.joblib successfully.')
print('Dataset SHA-256:', sha256)
print('Sample prediction:', restored['model'].predict([[5.1, 3.5, 1.4, 0.2]])[0])

Saved and reloaded models/model.joblib successfully.
Dataset SHA-256: 600ac44f23c2e6e0ae37daac8ceb2baba4df963efa580eb31b3b576b28e34c55
Sample prediction: Iris-setosa
